# 3주차. RAG와 Agentic RAG

## 0. 목표

학생이 직접 입력한 메모를 OpenAI embedding으로 ChromaDB에 넣고, 실제 Agent가 검색 도구를 호출해 답하게 만든다.


## 1. 준비

OpenAI API key와 ChromaDB가 필요하다.


In [ ]:
import json
import os
from typing import Any

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI

load_dotenv()

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
OPENAI_EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env 파일에 OPENAI_API_KEY를 설정한 뒤 다시 실행하세요.")


def make_model(max_tokens: int = 500) -> ChatOpenAI:
    return ChatOpenAI(model=OPENAI_MODEL, temperature=0, max_completion_tokens=max_tokens)


def show_json(value: Any) -> None:
    print(json.dumps(value, ensure_ascii=False, indent=2, default=str))


def final_text(agent_result: dict[str, Any]) -> str:
    return agent_result["messages"][-1].content


def extract_tool_trace(agent_result: dict[str, Any]) -> list[dict[str, Any]]:
    trace = []
    for message in agent_result.get("messages", []):
        for call in getattr(message, "tool_calls", []) or []:
            trace.append({"event": "tool_call", "tool_name": call.get("name"), "arguments": call.get("args", {})})
        if getattr(message, "type", None) == "tool":
            trace.append({"event": "tool_result", "tool_name": getattr(message, "name", None), "content": message.content})
    return trace

import uuid
import chromadb
from chromadb.config import Settings
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction


## 2. 개념

RAG는 모델이 모르는 정보를 검색 도구로 찾아 답변에 반영하는 흐름이다.


## 3. 기본 개념 실습

가장 작은 흐름은 학생이 직접 입력한 메모를 embedding collection에 저장하는 것이다. 아직 agent를 붙이지 않고 저장 상태만 확인한다.


In [ ]:
student_memories = [
    "프로젝트 발표는 2026-04-24 10:00에 민수와 지아가 함께 진행한다.",
    "카나메이트 UI에서는 채팅 답변과 tool trace를 함께 보여준다.",
]

embedding_function = OpenAIEmbeddingFunction(api_key=os.environ["OPENAI_API_KEY"], model_name=OPENAI_EMBEDDING_MODEL)
client = chromadb.Client(Settings(anonymized_telemetry=False))
memory_collection = client.create_collection(name=f"kanamate_week3_{uuid.uuid4().hex[:8]}", embedding_function=embedding_function)

memory_collection.add(
    ids=[f"memory-{index + 1}" for index in range(len(student_memories))],
    documents=student_memories,
    metadatas=[{"source": "student_input"} for _ in student_memories],
)

print("저장된 메모 수:", memory_collection.count())


## 4. 카나메이트 확장 예제

저장된 메모 검색 결과를 수업 코드가 쓰기 쉬운 `search_memory_hits` 리스트로 정리하고, 같은 helper를 agent tool에서 재사용한다.


In [ ]:
def search_memory_hits(query: str, top_k: int = 2) -> list[dict[str, Any]]:
    """Return Chroma search results as a simple list of dictionaries."""
    found = memory_collection.query(query_texts=[query], n_results=top_k)
    ids = found.get("ids", [[]])[0]
    documents = found.get("documents", [[]])[0]
    distances = found.get("distances", [[]])[0]
    return [
        {"id": ids[index], "content": documents[index], "distance": distances[index]}
        for index in range(len(ids))
    ]

@tool("search_memory", description="학생이 입력한 메모를 OpenAI embedding 기반으로 검색한다.")
def search_memory(query: str, top_k: int = 2) -> str:
    """Search student memory with OpenAI embeddings."""
    return json.dumps({"hits": search_memory_hits(query, top_k)}, ensure_ascii=False)

rag_agent = create_agent(
    model=make_model(700),
    tools=[search_memory],
    system_prompt="저장된 메모가 필요한 질문이면 search_memory 도구를 호출한 뒤, 찾은 근거를 바탕으로 답한다.",
)

student_question = "프로젝트 발표는 언제야?"
result = rag_agent.invoke({"messages": [{"role": "user", "content": student_question}]})

print(final_text(result))
show_json(extract_tool_trace(result))


## 5. 확장 예제 실행

같은 helper를 직접 호출해 hit 리스트를 보고, agent tool trace에도 같은 검색 결과가 들어가는지 확인한다.


In [ ]:
practice_question = "카나메이트 UI에서는 무엇을 함께 보여줘?"
practice_hits = search_memory_hits(practice_question, top_k=2)
practice_result = rag_agent.invoke({"messages": [{"role": "user", "content": practice_question}]})
practice_trace = extract_tool_trace(practice_result)

show_json(practice_hits)
print(final_text(practice_result))
show_json(practice_trace)

assert memory_collection.count() == len(student_memories)
assert practice_hits
assert any(event["event"] == "tool_call" and event["tool_name"] == "search_memory" for event in practice_trace)
assert any(event["event"] == "tool_result" and "hits" in event["content"] for event in practice_trace)
print("3주차 확장 예제 실행 통과")


## 6. 코드 작성형 실습(`kanamate_app.py`)

이번 실습은 `search_memory_hits` helper를 단일 파일 `kanamate_app.py`에서 바로 구현하고 실행한다. 노트북에서 입력한 `student_memories`를 다시 적재한 뒤 같은 검색 흐름을 확인한다.

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "kanamate_app.py").exists():
            return candidate
    raise RuntimeError("repo root를 찾지 못했습니다. 노트북을 repo 안에서 실행하세요.")


repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from kanamate_app import extract_tool_trace, final_text, show_json, build_week03_agent, reset_memory_collection, search_memory_hits
reset_memory_collection(student_memories)

### 실행 예시

In [ ]:
practice_question = "카나메이트 UI에서는 무엇을 함께 보여줘?"
practice_hits = search_memory_hits(practice_question, top_k=2)
practice_rag_agent = build_week03_agent(search_memory_hits)
practice_result = practice_rag_agent.invoke({"messages": [{"role": "user", "content": practice_question}]})

show_json(practice_hits)
print(final_text(practice_result))
show_json(extract_tool_trace(practice_result))

## 6-0. 실습 자동 점검

아래 셀은 모델 답변 문구가 아니라 trace, structured response, payload처럼 구조화된 값을 확인한다.

In [ ]:
practice_trace = extract_tool_trace(practice_result)
assert len(practice_hits) <= 2
assert practice_hits
assert all({"id", "content", "distance"}.issubset(hit.keys()) for hit in practice_hits)
assert any(event["event"] == "tool_call" and event["tool_name"] == "search_memory" for event in practice_trace)
print("3주차 코드 작성형 실습 통과")

## 6-1. 로컬 Gradio UI 실습

WebUI도 같은 단일 파일 `kanamate_app.py`에 들어 있다. 터미널에서 아래 명령을 실행하면 1-6주차 탭이 있는 Gradio UI가 바로 열린다.

```bash
python kanamate_app.py
```

노트북 안에서는 다음 셀에서 `create_demo()`로 Gradio Blocks 객체를 확인할 수 있다.

In [ ]:
from kanamate_app import create_demo

demo = create_demo()
demo

## 7. 회고

이번 주에는 학생이 직접 넣은 메모가 embedding 검색과 Agent tool call을 거쳐 답변에 들어가는 흐름을 확인했다.
